# 04-6. 최종 clean 데이터 노이즈 재분류

04-5의 summary 그룹 분할 완료 데이터를 입력으로 사용한다. 따라서 반복 4어절 행, URL·저작권 문구, 완전 중복, summary 기반 split 누수가 처리된 뒤 남은 데이터의 노이즈 후보를 검토용 CSV로 분류한다.

In [1]:
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
CLEAN_DIR = PROJECT / "processed_data" / "final_clean_summary_grouped"
REVIEW_DIR = PROJECT / "review_data" / "04-6_final_clean_candidates"
assert CLEAN_DIR.exists(), "먼저 04-5를 실행하세요."
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print("입력 clean 데이터:", CLEAN_DIR)
print("출력 후보 폴더:", REVIEW_DIR)

입력 clean 데이터: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\final_clean_summary_grouped
출력 후보 폴더: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-6_final_clean_candidates


In [2]:
frames = [pd.read_csv(CLEAN_DIR / f"{split}.csv") for split in ["train", "validation", "test"]]
all_df = pd.concat(frames, ignore_index=True)
assert all_df["row_id"].is_unique

word_pattern = re.compile(r"[가-힣A-Za-z0-9]+(?:[.,]\d+)?")

def normalized_words(text):
    return word_pattern.findall(str(text).lower())

def repeated_4gram_count(text):
    words = normalized_words(text)
    counts = Counter(tuple(words[i:i + 4]) for i in range(len(words) - 3))
    return max(counts.values(), default=0)

def overlap_ratio(document, summary):
    document_words = set(normalized_words(document))
    summary_words = normalized_words(summary)
    if not summary_words:
        return np.nan
    return sum(word in document_words for word in summary_words) / len(summary_words)

def normalized_numbers(text):
    text = re.sub(r"(?<=\d)\s+(?=\d)", "", str(text))
    return set(re.findall(r"\d+(?:[.,]\d+)?", text.replace(",", "")))

all_df["document_chars"] = all_df["document"].str.len()
all_df["summary_chars"] = all_df["summary"].str.len()
all_df["char_compression_ratio"] = all_df["summary_chars"] / all_df["document_chars"]
all_df["max_same_4gram_occurrences"] = all_df["summary"].map(repeated_4gram_count)
all_df["repeat_4gram"] = all_df["max_same_4gram_occurrences"] >= 3
all_df["summary_document_overlap"] = [
    overlap_ratio(document, summary) for document, summary in zip(all_df["document"], all_df["summary"])
]
all_df["low_overlap"] = all_df["summary_document_overlap"] < 0.30
all_df["extreme_ratio"] = (all_df["char_compression_ratio"] < 0.02) | (all_df["char_compression_ratio"] > 0.80)
all_df["summary_longer"] = all_df["summary_chars"] > all_df["document_chars"]
doc_numbers = all_df["document"].map(normalized_numbers)
summary_numbers = all_df["summary"].map(normalized_numbers)
all_df["numbers_only_in_summary"] = [", ".join(sorted(s - d)) for d, s in zip(doc_numbers, summary_numbers)]
all_df["number_mismatch_candidate"] = all_df["numbers_only_in_summary"].ne("")

assert not all_df["repeat_4gram"].any(), "04-2 제외 규칙과 04-6 재검토 기준이 일치하지 않습니다."
print("반복 4어절 재검출:", int(all_df["repeat_4gram"].sum()), "건")

반복 4어절 재검출: 0 건


In [3]:
BASE_COLUMNS = [
    "row_id", "split", "source_index", "date", "category", "press", "title", "link",
    "document", "summary", "max_same_4gram_occurrences", "summary_document_overlap",
    "document_chars", "summary_chars", "char_compression_ratio", "numbers_only_in_summary",
]
RULES = {
    "01_repeat_4gram_remaining": (all_df["repeat_4gram"], "정제 후 남은 4어절 3회 이상 반복; 0건이 정상"),
    "02_low_overlap": (all_df["low_overlap"], "요약 어절의 본문 등장 비율 30% 미만"),
    "03_extreme_ratio": (all_df["extreme_ratio"], "요약/본문 글자 비율 2% 미만 또는 80% 초과"),
    "04_summary_longer": (all_df["summary_longer"], "요약 글자 수가 본문보다 큼"),
    "05_number_mismatch": (all_df["number_mismatch_candidate"], "본문에 없는 숫자가 요약에 있음; 표기 차이 오탐 가능"),
}

def save_csv(frame, path):
    try:
        frame.to_csv(path, index=False, encoding="utf-8-sig")
        return path
    except PermissionError:
        fallback = path.with_name(path.stem + "_latest" + path.suffix)
        frame.to_csv(fallback, index=False, encoding="utf-8-sig")
        print("기존 CSV가 열려 있어 새 파일로 저장:", fallback.name)
        return fallback

exports = []
for name, (mask, description) in RULES.items():
    output = all_df.loc[mask, BASE_COLUMNS].copy()
    output.insert(3, "candidate_type", name)
    output.insert(4, "candidate_description", description)
    output["review_decision"] = ""
    output["drop_reason"] = ""
    output["memo"] = ""
    output_path = save_csv(output, REVIEW_DIR / f"{name}.csv")
    exports.append({"file": output_path.name, "rows": len(output), "description": description})

summary = pd.DataFrame(exports)
save_csv(summary, REVIEW_DIR / "00_export_summary.csv")
all_flagged = all_df.loc[all_df[["low_overlap", "extreme_ratio", "summary_longer", "number_mismatch_candidate"]].any(axis=1), BASE_COLUMNS + ["low_overlap", "extreme_ratio", "summary_longer", "number_mismatch_candidate"]].copy()
all_flagged["matched_rules"] = all_flagged[["low_overlap", "extreme_ratio", "summary_longer", "number_mismatch_candidate"]].apply(lambda row: ", ".join(row.index[row].tolist()), axis=1)
all_flagged["review_decision"] = ""
all_flagged["drop_reason"] = ""
all_flagged["memo"] = ""
save_csv(all_flagged, REVIEW_DIR / "00_all_remaining_flagged_rows.csv")

save_csv(pd.DataFrame([{
    "input_data": str(CLEAN_DIR),
    "repeat_rule": "same normalized 4-gram occurs at least 3 times; already excluded in 04-2",
    "input_rows": len(all_df),
    "remaining_repeat_4gram": int(all_df["repeat_4gram"].sum()),
}]), REVIEW_DIR / "00_run_info.csv")
display(summary)
print("여러 다른 노이즈에 해당하는 통합 후보:", f"{len(all_flagged):,}", "건")

,file,rows,description
0,01_repeat_4gram_remaining.csv,0,정제 후 남은 4어절 3회 이상 반복; 0건이 정상
1,02_low_overlap.csv,0,요약 어절의 본문 등장 비율 30% 미만
2,03_extreme_ratio.csv,1024,요약/본문 글자 비율 2% 미만 또는 80% 초과
3,04_summary_longer.csv,610,요약 글자 수가 본문보다 큼
4,05_number_mismatch.csv,643,본문에 없는 숫자가 요약에 있음; 표기 차이 오탐 가능


여러 다른 노이즈에 해당하는 통합 후보: 1,617 건


`01_repeat_4gram_remaining.csv`가 0건이면 04-2의 제외가 올바르게 반영된 것이다. 이후에는 `02`~`05`와 `00_all_remaining_flagged_rows.csv`만 검토하여 다음 전처리 규칙을 정한다.